# ChronoCell-5D · GPU reconstruction on Colab (T4)

Turns Micro-C contacts + H3K27ac + sequence into **3D spatial coordinates** for a human chromosome, for one or more **conditions** (e.g. healthy vs tumour, control vs Parkinson's iPSC neurons), each with one or more **frames** (time points) — the 4th dimension. The output drops straight into the workstation's coordinate slot (`coordinates/<chrom>/`).

**Before running:** *Runtime → Change runtime type → T4 GPU*. Then run the cells top to bottom.

`DEMO = True` runs the whole pipeline on planted synthetic data (no downloads) so you can check it end to end. Set `DEMO = False` and fill in your files to run on real data.

In [ ]:
# 1 · GPU check
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> T4 GPU"
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2 · Get the ChronoCell code
Upload `colab/chronocell_code.zip` from the project (create it with `python colab/pack_code.py`). Alternatively mount Google Drive and copy the project folder.

In [ ]:
import os, zipfile
if not os.path.isdir("chronocell"):
    from google.colab import files
    uploaded = files.upload()                       # choose chronocell_code.zip
    for name in uploaded:
        zipfile.ZipFile(name).extractall(".")
# Google Drive alternative:
# from google.colab import drive; drive.mount('/content/drive'); !cp -r "/content/drive/MyDrive/ChronoCell-5D/chronocell" .
!pip -q install pyBigWig cooler pyfaidx
from chronocell import colab, egnn, genome, physics, formats
print(colab.environment())

## 3 · Configure
* `CHROM`: any of chr1…chr22, chrX, chrY. `RESOLUTION = None` picks the finest standard bin size giving ≤ 6,000 beads (chr22 → 10 kb, chr1 → 50 kb). The `.mcool` must contain that resolution.
* One entry per **condition**, one **Frame** per time point. Local paths or `https://` URLs.
* Where to find data: Micro-C / Hi-C `.mcool` files on the **4DN Data Portal** (data.4dnucleome.org); H3K27ac fold-change `.bigWig` on the **ENCODE portal** (encodeproject.org). Disease-relevant examples: K562 (CML line carrying BCR–ABL1) Hi-C/Micro-C for cancer; iPSC-derived dopaminergic neurons for Parkinson's studies, where available.

In [ ]:
DEMO = True
CHROM = "chr22"
RESOLUTION = None
DEMO_FRAMES = 3

CONDITIONS = [
    colab.Condition("healthy", [
        colab.Frame("0 h",  0,  mcool="https://…/control_t0.mcool",  bigwig="https://…/control_H3K27ac.bigWig"),
        colab.Frame("24 h", 24, mcool="https://…/control_t24.mcool", bigwig="https://…/control_H3K27ac.bigWig"),
    ]),
    colab.Condition("tumour", [
        colab.Frame("0 h",  0,  mcool="https://…/tumour_t0.mcool",   bigwig="https://…/tumour_H3K27ac.bigWig"),
    ]),
]

# Whole chromosome on a T4: 2,500 epochs (~1-2 min). Benchmarked on a planted chr22:
# assembled-bead RMSD 0.32 Rg, distance correlation 0.93 (APP_GUIDE.md, 'Accuracy').
FIT = egnn.FitConfig(prefit_epochs=2500, refine_epochs=100, device="auto")       # cold start (first frame)
WARM = egnn.FitConfig(prefit_epochs=300, phantom_fraction=0.0, lr_prefit=0.02,   # later frames, warm-started
                      refine_epochs=100, device="auto")
chrom = genome.chrom(CHROM, RESOLUTION)
print(chrom.name, f"{chrom.resolution:,} bp bins,", f"{chrom.n_bins:,} beads, b0 = {physics.bond_length_for(chrom.resolution):.0f} nm")

In [ ]:
# 4 · Inputs -> graphs (GC, H3K27ac, Micro-C contacts per frame)
import urllib.request
def local(path):
    if path and path.startswith("http"):
        dest = os.path.join("inputs", path.split("/")[-1].split("?")[0])
        os.makedirs("inputs", exist_ok=True)
        if not os.path.exists(dest):
            print("downloading", path); urllib.request.urlretrieve(path, dest)
        return dest
    return path

graphs = {}
if DEMO:
    graphs = colab.demo_graphs(chrom, n_frames=DEMO_FRAMES)
    CONDITIONS = [colab.Condition(name, [colab.Frame(f"{24 * t} h", 24 * t) for t in range(DEMO_FRAMES)])
                  for name in graphs]
else:
    fasta = colab.download_fasta(CHROM, "inputs")                      # UCSC hg38 sequence
    for cond in CONDITIONS:
        graphs[cond.name] = []
        for fr in cond.frames:
            fr.mcool, fr.bigwig = local(fr.mcool), local(fr.bigwig)
            graphs[cond.name].append(colab.build_graph(CHROM, chrom.resolution, fasta, fr))
for name, gs in graphs.items():
    print(name, [f"{len(g.ci):,} contacts" for g in gs])

In [ ]:
# 5 · Reconstruct every frame of every condition on the GPU
results = [colab.reconstruct_condition(chrom, graphs[c.name], c, cfg=FIT, warm_cfg=WARM) for c in CONDITIONS]

In [ ]:
# 6 · Quality control
from chronocell import scenarios
for r in results:
    m = scenarios.frame_metrics(scenarios.from_bundle(r.frames_nm, r.times, r.labels, chrom, 0, r.condition, "", "hours"))
    fit = physics.distance_scaling(r.frames_nm[-1])
    ov = physics.loss_steric(r.frames_nm[-1], b0=physics.bond_length_for(chrom.resolution)).overlaps
    print(f"{r.condition:12s} Rg {m['rg'].round(0)} nm | RMSD vs first frame {m['rmsd'].round(1)} nm | "
          f"nu {fit.nu:.3f} ({fit.regime}) | overlaps {ov} | L_contact {[round(v, 4) for v in r.final_contact_loss]}")
print("E(3) equivariance:", {k: f"{v:.1e}" for k, v in egnn.equivariance_check().items()})

import plotly.graph_objects as go
x = results[0].frames_nm[-1]
go.Figure(go.Scatter3d(x=x[:, 0], y=x[:, 1], z=x[:, 2], mode="lines", line=dict(color=list(range(len(x))), width=4,
          colorscale="Viridis"))).update_layout(height=520, scene=dict(aspectmode="data")).show()

In [ ]:
# 7 · Write coordinates/<chrom>/<condition>.npz (+ graph.npz) and download the zip
zpath = colab.write_outputs(chrom, results, "output", source="Colab " + ("demo" if DEMO else "Micro-C"))
print("wrote", zpath)
from google.colab import files
files.download(zpath)
# or keep it on Drive:  !cp "{zpath}" /content/drive/MyDrive/

## 8 · Load into the workstation
Unzip the downloaded file at the project root: it creates `coordinates/<chrom>/…`. Restart or refresh `streamlit run app.py`; the reference model is replaced automatically.
* **3D structure** shows each condition (Data → Structure source) and its frames (Frame slider).
* **4D dynamics** plays *Provided frames* (time course) or *Across conditions* (e.g. healthy → tumour), with R<sub>g</sub>, RMSD and per-bead displacement over time.